In [ ]:
# -*- coding: utf-8 -*-
"""
RF PONTO A PONTO — COMPARAÇÃO DA LARGURA DAS FAIXAS POR TEMPERATURA
=====================================================================

Este código foi feito para manter a MESMA lógica de Random Forest ponto a ponto
usada no seu código original de varredura RF vs Park, mas removendo totalmente
a parte do Park.

O que este código faz:

1) Lê a mesma base do código original:
      ARQ_BASE = "base-completo--.pkl"

2) Usa os mesmos parâmetros principais do RF ponto a ponto:
      REF_TEMP = 30
      SMOOTH_WIN = 5
      RF_COMP_POINT_PARAMS igual ao código original

3) Testa faixas acumuladas começando em 30 kHz:
      30–40 kHz
      30–50 kHz
      30–60 kHz
      30–70 kHz
      30–80 kHz
      30–90 kHz
      30–100 kHz

4) Para CADA faixa:
      - seleciona as colunas f_XXXXXHz daquela faixa
      - monta a curva referência saudável DAQUELA MESMA FAIXA
      - treina o RF ponto a ponto usando apenas falha == 0
      - compensa todas as curvas
      - calcula RMSD e CCDM comparando com a referência saudável correta daquela faixa

5) Para CADA temperatura alvo de -10 até 80 °C:
      - se a temperatura exata existir e tiver danos 0, 1 e 2, usa ela
      - se não existir exatamente, usa a temperatura mais próxima que tenha danos 0, 1 e 2
      - gera um gráfico com RMSD e CCDM lado a lado
      - compara as larguras das faixas no eixo x
      - barras separadas para dano 0, dano 1 e dano 2

Arquivos salvos:

resultados_rf_ponto_a_ponto_larguras_temperaturas_CORRIGIDO/
    metricas_amostra_a_amostra_rf_todas_larguras.csv
    resumo_metricas_rf_por_faixa_temperatura_dano.csv
    resumo_metricas_rf_todas_larguras_geral.csv
    referencias_usadas_por_faixa.csv
    mapa_temperaturas_alvo_usadas.csv
    graficos/
        rf_larguras_Talvo_...png/pdf

PONTO MAIS IMPORTANTE:
---------------------
O RMSD e o CCDM são calculados contra y_ref da própria faixa analisada.
Ou seja, a faixa 30–40 usa uma referência saudável 30–40.
A faixa 30–100 usa uma referência saudável 30–100.
Assim o gráfico condiz com o resultado real do RF naquela faixa.
"""

# ============================================================
# 1) IMPORTS
# ============================================================
import os
import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# 2) PARÂMETROS GERAIS — MESMOS DO CÓDIGO ORIGINAL NA PARTE RF
# ============================================================

ARQ_BASE = "base-completo--.pkl"

# Temperatura de referência usada para montar a curva referência saudável
REF_TEMP = 30

# Pasta de saída
PASTA_SAIDA = "resultados_rf_ponto_a_ponto_larguras_temperaturas_CORRIGIDO"
PASTA_GRAFICOS = os.path.join(PASTA_SAIDA, "graficos")
os.makedirs(PASTA_SAIDA, exist_ok=True)
os.makedirs(PASTA_GRAFICOS, exist_ok=True)

# Suavização das curvas compensadas
SMOOTH_WIN = 5

# Random Forest ponto a ponto — MESMOS PARÂMETROS DO SEU CÓDIGO
RF_COMP_POINT_PARAMS = dict(
    n_estimators=250,
    max_depth=10,
    min_samples_leaf=2,
    min_samples_split=4,
    max_features="sqrt",
    n_jobs=-1,
    random_state=0,
)

# Faixas acumuladas que você pediu: 30-40, 30-50, ..., 30-100 kHz
FREQ_INICIO_FIXO_KHZ = 30
FREQ_MAXIMOS_KHZ = list(range(40, 101, 10))

# Temperaturas alvo: -10, 0, 10, ..., 80 °C
TEMPERATURAS_ALVO = list(range(-10, 81, 10))

# Danos que devem aparecer nos gráficos
DANOS_PLOTAR = [0, 1, 2]

# Salvar também em PDF
SALVAR_PDF = True


# ============================================================
# 3) CONFIGURAÇÃO VISUAL DOS GRÁFICOS
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 22,
    "axes.labelsize": 26,
    "axes.titlesize": 26,
    "xtick.labelsize": 18,
    "ytick.labelsize": 22,
    "legend.fontsize": 18,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

CORES_DANO = {
    0: "tab:blue",
    1: "tab:orange",
    2: "tab:red",
}


# ============================================================
# 4) FUNÇÕES BÁSICAS — MANTIDAS DO CÓDIGO ORIGINAL
# ============================================================

def extract_freq_hz(col):
    """Extrai a frequência de colunas no formato f_30000Hz."""
    m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
    return float(m.group(1)) if m else None


def get_freq_columns(df, fmin_khz, fmax_khz):
    """Seleciona colunas de frequência dentro da faixa escolhida."""
    cols, freqs = [], []

    for c in df.columns:
        f = extract_freq_hz(c)
        if f is not None:
            f_khz = f / 1e3
            if fmin_khz <= f_khz <= fmax_khz:
                cols.append(c)
                freqs.append(f)

    if len(cols) == 0:
        return [], np.array([])

    order = np.argsort(freqs)
    cols = [cols[i] for i in order]
    freqs = np.array(freqs)[order]

    return cols, freqs


def moving_average(arr, win):
    """Média móvel simples para suavizar a curva."""
    arr = np.asarray(arr, dtype=float)

    if win <= 1 or win % 2 == 0:
        return arr.copy()

    pad = win // 2
    arr_pad = np.pad(arr, (pad, pad), mode="edge")
    kernel = np.ones(win) / win
    smooth = np.convolve(arr_pad, kernel, mode="valid")

    if len(smooth) > len(arr):
        smooth = smooth[:len(arr)]
    elif len(smooth) < len(arr):
        smooth = np.pad(smooth, (0, len(arr) - len(smooth)), mode="edge")

    return smooth


def add_extra_features_matrix(X):
    """
    Mantém a lógica do seu RF ponto a ponto:
    usa a curva inteira + média + desvio padrão + amplitude.
    """
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True)
    amp = (X.max(axis=1) - X.min(axis=1)).reshape(-1, 1)
    return np.hstack([X, mu, sd, amp])


def rmsd(y, ref):
    """RMSD: quanto menor, mais perto da referência."""
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)
    return float(np.sqrt(np.mean((y - ref) ** 2)))


def ccdm(y, ref):
    """
    CCDM = 1 - correlação de Pearson.
    Quanto menor, mais parecida é a forma da curva.
    """
    y = np.asarray(y, dtype=float)
    ref = np.asarray(ref, dtype=float)

    y0 = y - np.mean(y)
    r0 = ref - np.mean(ref)

    num = float(np.sum(y0 * r0))
    den = float(np.sqrt(np.sum(y0 ** 2) * np.sum(r0 ** 2))) + 1e-18

    corr = num / den
    return float(1 - corr)


def curva_referencia_saudavel(df, fcols):
    """
    Curva de referência = mediana das curvas sem dano na temperatura REF_TEMP.
    Se não existir REF_TEMP, usa a mediana de todas as curvas sem dano.

    ATENÇÃO:
    Esta função recebe fcols da faixa atual.
    Então a referência sempre tem o mesmo tamanho da faixa analisada.
    """
    df_sem = df[df["falha"] == 0]
    X_sem = df_sem[fcols].to_numpy(float)

    pool = df_sem.loc[np.isclose(df_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)

    if len(pool) > 0:
        return np.median(pool, axis=0)

    print(f"⚠️ Não achei dados sem dano em {REF_TEMP}°C. Usando mediana geral sem dano.")
    return np.median(X_sem, axis=0)


def info_referencia_saudavel(df):
    """
    Apenas registra de onde veio a referência saudável.
    Não muda o cálculo original.
    """
    df_sem = df[df["falha"] == 0]
    n_sem_total = len(df_sem)
    n_sem_ref = int(np.isclose(df_sem["temperatura_c"], REF_TEMP).sum())

    if n_sem_ref > 0:
        modo = f"mediana_falha0_{REF_TEMP}C"
        n_usado = n_sem_ref
    else:
        modo = "mediana_falha0_todas_temperaturas"
        n_usado = n_sem_total

    return modo, n_usado, n_sem_total


# ============================================================
# 5) MÉTODO — RF DIRETO PONTO A PONTO
# ============================================================

def compensar_rf_direto(df, fcols):
    """
    Treina o RF apenas com dados sem dano.
    Entrada: curva medida + features simples + temperatura.
    Saída aprendida: correção necessária para levar a curva sem dano até a referência.

    Esta função é a parte importante do seu código original.
    Ela foi mantida com a mesma estrutura.
    """
    df_sem = df[df["falha"] == 0]

    X_sem = df_sem[fcols].to_numpy(float)
    T_sem = df_sem["temperatura_c"].to_numpy(float)

    y_ref = curva_referencia_saudavel(df, fcols)

    # O alvo é o resíduo térmico aprendido nos dados saudáveis
    Y_target = y_ref[None, :] - X_sem

    X_aug = add_extra_features_matrix(X_sem)
    X_in = np.hstack([X_aug, T_sem.reshape(-1, 1)])

    rf = RandomForestRegressor(**RF_COMP_POINT_PARAMS)
    rf.fit(X_in, Y_target)

    X_all = df[fcols].to_numpy(float)
    T_all = df["temperatura_c"].to_numpy(float)

    X_aug_all = add_extra_features_matrix(X_all)
    X_in_all = np.hstack([X_aug_all, T_all.reshape(-1, 1)])

    Y_comp = X_all + rf.predict(X_in_all)

    for i in range(len(Y_comp)):
        Y_comp[i] = moving_average(Y_comp[i], SMOOTH_WIN)

    df2 = df.copy()
    df2[fcols] = Y_comp

    return df2, y_ref


# ============================================================
# 6) MÉTRICAS — COMPARAÇÃO COM A REFERÊNCIA CORRETA DA FAIXA
# ============================================================

def calcular_metricas(df_comp, fcols, y_ref):
    """
    Calcula RMSD e CCDM de cada curva compensada contra y_ref.

    ATENÇÃO:
    y_ref é a referência saudável calculada para a MESMA faixa fcols.
    Isso evita comparar 30-40 kHz com referência de outra largura.
    """
    X = df_comp[fcols].to_numpy(float)

    df2 = df_comp[["temperatura_c", "falha"]].copy()
    df2["RMSD"] = [rmsd(x, y_ref) for x in X]
    df2["CCDM"] = [ccdm(x, y_ref) for x in X]

    return df2


def resumir_metricas_por_faixa_temperatura_dano(df_metricas):
    """Resumo por faixa, temperatura e dano."""
    resumo = (
        df_metricas
        .groupby(["faixa_min_khz", "faixa_max_khz", "largura_khz", "faixa_label", "temperatura_c", "falha"], as_index=False)
        .agg(
            RMSD_medio=("RMSD", "mean"),
            RMSD_std=("RMSD", "std"),
            CCDM_medio=("CCDM", "mean"),
            CCDM_std=("CCDM", "std"),
            n_amostras=("RMSD", "size"),
        )
    )
    return resumo


def resumir_metricas_geral_por_dano(df_metricas):
    """Resumo geral por faixa e dano, juntando todas as temperaturas."""
    resumo = (
        df_metricas
        .groupby(["faixa_min_khz", "faixa_max_khz", "largura_khz", "faixa_label", "falha"], as_index=False)
        .agg(
            RMSD_medio=("RMSD", "mean"),
            RMSD_std=("RMSD", "std"),
            CCDM_medio=("CCDM", "mean"),
            CCDM_std=("CCDM", "std"),
            n_amostras=("RMSD", "size"),
        )
    )
    return resumo


# ============================================================
# 7) TEMPERATURA MAIS PRÓXIMA COM OS TRÊS DANOS
# ============================================================

def temperaturas_com_todos_danos(df_base, danos=DANOS_PLOTAR):
    """
    Retorna temperaturas que possuem pelo menos uma amostra de cada dano pedido.
    Isso evita plotar D0 em uma temperatura e D1/D2 em outra sem perceber.
    """
    temps_validas = []

    for T in sorted(df_base["temperatura_c"].dropna().unique()):
        ok = True
        for d in danos:
            mask = np.isclose(df_base["temperatura_c"], T) & (df_base["falha"] == d)
            if not np.any(mask):
                ok = False
                break
        if ok:
            temps_validas.append(float(T))

    return temps_validas


def escolher_temperatura_mais_proxima(temps_validas, temperatura_alvo):
    """
    Se a temperatura alvo existir, usa ela.
    Caso contrário, usa a temperatura válida mais próxima.
    """
    if len(temps_validas) == 0:
        raise ValueError("Não existe nenhuma temperatura com todos os danos pedidos.")

    temps = np.asarray(temps_validas, dtype=float)
    idx = int(np.argmin(np.abs(temps - temperatura_alvo)))
    return float(temps[idx])


def montar_mapa_temperaturas(df_base):
    """Monta tabela alvo -> temperatura realmente usada."""
    temps_validas = temperaturas_com_todos_danos(df_base, DANOS_PLOTAR)

    rows = []
    for T_alvo in TEMPERATURAS_ALVO:
        T_usada = escolher_temperatura_mais_proxima(temps_validas, T_alvo)
        rows.append({
            "temperatura_alvo_c": float(T_alvo),
            "temperatura_usada_c": float(T_usada),
            "diferenca_abs_c": float(abs(T_usada - T_alvo)),
            "exata": bool(np.isclose(T_usada, T_alvo)),
        })

    return pd.DataFrame(rows)


# ============================================================
# 8) EXECUTAR UMA FAIXA RF
# ============================================================

def executar_uma_faixa_rf(df_base, fmin_khz, fmax_khz):
    """Roda APENAS RF ponto a ponto para uma faixa específica."""

    fcols, fHz = get_freq_columns(df_base, fmin_khz, fmax_khz)

    if len(fcols) < 5:
        raise ValueError(f"Poucas colunas na faixa {fmin_khz}-{fmax_khz} kHz.")

    df_use = df_base[["temperatura_c", "falha"] + fcols].copy()

    print(f"\n🔹 RF ponto a ponto | faixa {fmin_khz:.0f}-{fmax_khz:.0f} kHz | {len(fcols)} pontos")

    modo_ref, n_ref_usado, n_sem_total = info_referencia_saudavel(df_use)
    print(f"   Referência usada: {modo_ref} | n_ref = {n_ref_usado} | n_sem_dano_total = {n_sem_total}")

    t0 = time.time()

    df_rf_comp, y_ref = compensar_rf_direto(df_use, fcols)
    df_rf_met = calcular_metricas(df_rf_comp, fcols, y_ref)

    # Informações da faixa
    df_rf_met["faixa_min_khz"] = float(fmin_khz)
    df_rf_met["faixa_max_khz"] = float(fmax_khz)
    df_rf_met["largura_khz"] = float(fmax_khz - fmin_khz)
    df_rf_met["faixa_label"] = f"{int(fmin_khz)}–{int(fmax_khz)} kHz"
    df_rf_met["n_freq_points"] = int(len(fcols))
    df_rf_met["metodo"] = "RF_ponto_a_ponto"

    # Checagem de segurança para garantir que a referência bate com a faixa
    if len(y_ref) != len(fcols):
        raise RuntimeError(
            f"Erro grave: y_ref tem tamanho {len(y_ref)}, mas a faixa tem {len(fcols)} pontos."
        )

    dt = time.time() - t0
    print(f"✅ Concluído em {dt:.1f} s")

    info_ref = {
        "faixa_min_khz": float(fmin_khz),
        "faixa_max_khz": float(fmax_khz),
        "largura_khz": float(fmax_khz - fmin_khz),
        "faixa_label": f"{int(fmin_khz)}–{int(fmax_khz)} kHz",
        "n_freq_points": int(len(fcols)),
        "freq_min_real_hz": float(np.min(fHz)),
        "freq_max_real_hz": float(np.max(fHz)),
        "ref_temp_c": float(REF_TEMP),
        "modo_referencia": modo_ref,
        "n_curvas_referencia_usadas": int(n_ref_usado),
        "n_curvas_sem_dano_total": int(n_sem_total),
        "y_ref_tamanho": int(len(y_ref)),
    }

    return df_rf_met, info_ref


# ============================================================
# 9) PLOT — TEMPERATURA FIXA, COMPARANDO LARGURAS DAS FAIXAS
# ============================================================

def _format_temp(T):
    T = float(T)
    if float(T).is_integer():
        return f"{int(T)}"
    return f"{T:.1f}"


def _pegar_valor_metricas(df_metricas, fmin, fmax, T_usada, dano, metrica):
    mask = (
        np.isclose(df_metricas["faixa_min_khz"], fmin) &
        np.isclose(df_metricas["faixa_max_khz"], fmax) &
        np.isclose(df_metricas["temperatura_c"], T_usada) &
        (df_metricas["falha"] == dano)
    )

    vals = df_metricas.loc[mask, metrica].to_numpy(float)

    if len(vals) == 0:
        return np.nan, np.nan, 0

    return float(np.nanmean(vals)), float(np.nanstd(vals, ddof=1)) if len(vals) > 1 else 0.0, int(len(vals))


def plotar_temperatura_comparando_larguras(df_metricas, temperatura_alvo, temperatura_usada):
    """
    Gera gráfico com dois painéis:
    - RMSD
    - CCDM

    Eixo x: 30–40, 30–50, ..., 30–100 kHz
    Barras: dano 0, dano 1 e dano 2
    """

    faixas_df = (
        df_metricas[["faixa_min_khz", "faixa_max_khz", "largura_khz", "faixa_label"]]
        .drop_duplicates()
        .sort_values(["faixa_min_khz", "faixa_max_khz"])
        .reset_index(drop=True)
    )

    labels_faixas = faixas_df["faixa_label"].tolist()
    x = np.arange(len(labels_faixas))

    bar_w = 0.24
    offsets = {
        0: -bar_w,
        1: 0.0,
        2: bar_w,
    }

    fig, axes = plt.subplots(1, 2, figsize=(24, 8.5), dpi=300)

    for ax in axes:
        ax.grid(False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="both", labelsize=20)
        ax.set_xticks(x)
        ax.set_xticklabels(labels_faixas, rotation=35, ha="right")
        ax.set_xlabel("Faixa de frequência analisada", fontsize=26, labelpad=12)

    metricas = ["RMSD", "CCDM"]
    titulos = ["(a) RMSD", "(b) CCDM"]

    for ax, metrica, titulo in zip(axes, metricas, titulos):
        for dano in DANOS_PLOTAR:
            medias = []
            erros = []

            for _, row in faixas_df.iterrows():
                media, std, n = _pegar_valor_metricas(
                    df_metricas=df_metricas,
                    fmin=float(row["faixa_min_khz"]),
                    fmax=float(row["faixa_max_khz"]),
                    T_usada=float(temperatura_usada),
                    dano=int(dano),
                    metrica=metrica,
                )
                medias.append(media)
                erros.append(std)

            ax.bar(
                x + offsets[dano],
                medias,
                yerr=erros,
                capsize=4,
                width=bar_w,
                color=CORES_DANO.get(dano, None),
                edgecolor="black",
                linewidth=0.9,
                label=f"Dano {dano}",
            )

        ax.set_ylabel(metrica, fontsize=28)
        ax.set_title(titulo, fontsize=28, pad=16)

    if np.isclose(temperatura_alvo, temperatura_usada):
        subtitulo_temp = f"Temperatura { _format_temp(temperatura_usada) }°C"
    else:
        subtitulo_temp = (
            f"Temperatura alvo { _format_temp(temperatura_alvo) }°C "
            f"| usada { _format_temp(temperatura_usada) }°C"
        )

    fig.suptitle(
        f"RF ponto a ponto — comparação da largura das faixas — {subtitulo_temp} — referência {REF_TEMP}°C",
        fontsize=30,
        y=1.02,
    )

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=3,
        frameon=True,
        fontsize=20,
        bbox_to_anchor=(0.5, -0.08),
    )

    fig.tight_layout(rect=[0, 0.08, 1, 0.95])

    nome_base = (
        f"rf_larguras_Talvo_{_format_temp(temperatura_alvo).replace('-', 'm')}C_"
        f"Usada_{_format_temp(temperatura_usada).replace('-', 'm')}C"
    )

    png_path = os.path.join(PASTA_GRAFICOS, nome_base + ".png")
    fig.savefig(png_path, dpi=600, bbox_inches="tight")

    if SALVAR_PDF:
        pdf_path = os.path.join(PASTA_GRAFICOS, nome_base + ".pdf")
        fig.savefig(pdf_path, bbox_inches="tight")
        print(f"✅ Gráfico salvo:\n{png_path}\n{pdf_path}")
    else:
        print(f"✅ Gráfico salvo:\n{png_path}")

    plt.show()

    return fig


# ============================================================
# 10) EXECUTAR TUDO
# ============================================================

def carregar_base():
    print("🔹 Carregando base...")
    print(f"Arquivo: {ARQ_BASE}")

    if not os.path.exists(ARQ_BASE):
        raise FileNotFoundError(
            f"Não encontrei o arquivo {ARQ_BASE}.\n"
            "Coloque este script na mesma pasta do base-completo--.pkl ou ajuste ARQ_BASE."
        )

    df_base = pd.read_pickle(ARQ_BASE)

    required = ["temperatura_c", "falha"]
    missing = [c for c in required if c not in df_base.columns]
    if missing:
        raise ValueError(f"A base não tem as colunas obrigatórias: {missing}")

    df_base = df_base.copy()
    df_base["temperatura_c"] = pd.to_numeric(df_base["temperatura_c"], errors="coerce")
    df_base["falha"] = pd.to_numeric(df_base["falha"], errors="coerce").astype(int)

    n_freq = sum(extract_freq_hz(c) is not None for c in df_base.columns)

    print(f"✅ Base carregada: {df_base.shape[0]} amostras | {n_freq} pontos de frequência")
    print(f"Temperaturas disponíveis: {sorted(df_base['temperatura_c'].dropna().unique())}")
    print(f"Danos disponíveis: {sorted(df_base['falha'].dropna().unique())}")

    return df_base


def rodar_rf_larguras_temperaturas():
    df_base = carregar_base()

    # Mapa alvo -> temperatura usada
    df_mapa_temps = montar_mapa_temperaturas(df_base)
    path_mapa = os.path.join(PASTA_SAIDA, "mapa_temperaturas_alvo_usadas.csv")
    df_mapa_temps.to_csv(path_mapa, index=False)

    print("\n📌 Mapa de temperaturas alvo -> temperatura usada:")
    print(df_mapa_temps.to_string(index=False))
    print(f"Mapa salvo em: {path_mapa}")

    # Rodar RF para cada largura de faixa
    todas_metricas = []
    referencias_info = []
    erros = []

    faixas = [(float(FREQ_INICIO_FIXO_KHZ), float(fmax)) for fmax in FREQ_MAXIMOS_KHZ]

    print("\n🔎 Faixas que serão analisadas:")
    for fmin, fmax in faixas:
        print(f"   {fmin:.0f}–{fmax:.0f} kHz")

    for fmin, fmax in faixas:
        try:
            df_met, info_ref = executar_uma_faixa_rf(df_base, fmin, fmax)
            todas_metricas.append(df_met)
            referencias_info.append(info_ref)

        except Exception as e:
            print(f"⚠️ Erro na faixa {fmin:.0f}-{fmax:.0f} kHz: {e}")
            erros.append({
                "faixa_min_khz": fmin,
                "faixa_max_khz": fmax,
                "erro": str(e),
            })

    if len(todas_metricas) == 0:
        raise RuntimeError("Nenhuma faixa foi processada com sucesso.")

    df_metricas = pd.concat(todas_metricas, ignore_index=True)
    df_refs = pd.DataFrame(referencias_info)

    df_resumo_temp = resumir_metricas_por_faixa_temperatura_dano(df_metricas)
    df_resumo_geral = resumir_metricas_geral_por_dano(df_metricas)

    # Salvar CSVs
    path_metricas = os.path.join(PASTA_SAIDA, "metricas_amostra_a_amostra_rf_todas_larguras.csv")
    path_resumo_temp = os.path.join(PASTA_SAIDA, "resumo_metricas_rf_por_faixa_temperatura_dano.csv")
    path_resumo_geral = os.path.join(PASTA_SAIDA, "resumo_metricas_rf_todas_larguras_geral.csv")
    path_refs = os.path.join(PASTA_SAIDA, "referencias_usadas_por_faixa.csv")

    df_metricas.to_csv(path_metricas, index=False)
    df_resumo_temp.to_csv(path_resumo_temp, index=False)
    df_resumo_geral.to_csv(path_resumo_geral, index=False)
    df_refs.to_csv(path_refs, index=False)

    print("\n✅ CSVs salvos:")
    print(path_metricas)
    print(path_resumo_temp)
    print(path_resumo_geral)
    print(path_refs)

    if len(erros) > 0:
        path_erros = os.path.join(PASTA_SAIDA, "erros_rf_larguras.csv")
        pd.DataFrame(erros).to_csv(path_erros, index=False)
        print(f"⚠️ Alguns erros foram salvos em: {path_erros}")

    # Gerar gráficos para todas as temperaturas alvo
    print("\n📊 Gerando gráficos por temperatura...")

    for _, row in df_mapa_temps.iterrows():
        T_alvo = float(row["temperatura_alvo_c"])
        T_usada = float(row["temperatura_usada_c"])
        plotar_temperatura_comparando_larguras(df_metricas, T_alvo, T_usada)

    print("\n✅ Tudo finalizado.")
    print(f"📁 Pasta de saída: {PASTA_SAIDA}")
    print(f"📁 Pasta dos gráficos: {PASTA_GRAFICOS}")

    return df_base, df_metricas, df_resumo_temp, df_resumo_geral, df_mapa_temps, df_refs


# ============================================================
# 11) EXECUÇÃO
# ============================================================

if __name__ == "__main__":
    (
        df_base,
        df_metricas_rf,
        df_resumo_por_temp,
        df_resumo_geral,
        df_mapa_temperaturas,
        df_referencias,
    ) = rodar_rf_larguras_temperaturas()


In [ ]:
# -*- coding: utf-8 -*-
"""
GRÁFICOS EXTRAS V2 — PESQUISA RF / FAIXAS / TEMPERATURAS / PARK
==================================================================

COMO USAR
---------
Cole este bloco NO FINAL do último código RF corrigido, ou rode separado depois
que o último código já tiver gerado o CSV:

    metricas_amostra_a_amostra_rf_todas_larguras.csv

Este script NÃO altera a lógica do RF. Ele apenas lê as métricas já calculadas
com a mesma referência correta de cada faixa e gera gráficos melhores para
pesquisa, com fontes ajustadas para não sobrepor.

MELHORIAS DA V2
---------------
1) Temperaturas organizadas em alvos fixos de -10 a 80 °C.
   Se uma temperatura exata não existir, usa a temperatura mais próxima.

2) Heatmaps menos poluídos:
   - menos sobreposição de fonte;
   - tamanho automático da figura;
   - valores dentro das células opcionais;
   - uma versão tripla e versões individuais por dano.

3) Novos gráficos de pesquisa:
   - RMSD/CCDM por temperatura × largura de faixa;
   - separação D1-D0, D2-D1 e D2-D0;
   - contraste relativo D2/D0 e D1/D0;
   - índice de separabilidade com penalização de inversão;
   - mapa de inversões D0 < D1 < D2;
   - ranking médio das faixas;
   - melhor faixa por temperatura;
   - estabilidade térmica por faixa;
   - sensibilidade térmica, isto é, inclinação da métrica contra |T-Tref|;
   - gráfico Pareto: erro no sem dano × separação de dano;
   - comparação RF × Park, se você ativar.

OBSERVAÇÃO IMPORTANTE
---------------------
Para pesquisa, não olhe só RMSD/CCDM baixos. Uma faixa boa deve:
    - deixar dano 0 perto da referência;
    - manter dano 1 e dano 2 separados do dano 0;
    - evitar inversões tipo D2 menor que D1;
    - ser estável quando a temperatura muda.
"""

# ============================================================
# 1) IMPORTS
# ============================================================
import os
import re
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)


# ============================================================
# 2) CONFIGURAÇÕES PRINCIPAIS
# ============================================================

# Se este bloco for colado no final do último código, ele reaproveita variáveis globais.
ARQ_BASE = globals().get("ARQ_BASE", "base-completo--.pkl")
REF_TEMP = globals().get("REF_TEMP", 30)
SMOOTH_WIN = globals().get("SMOOTH_WIN", 5)

# Pasta do último código RF corrigido.
PASTA_RESULTADOS_RF = globals().get(
    "PASTA_SAIDA",
    "resultados_rf_ponto_a_ponto_larguras_temperaturas_CORRIGIDO"
)

# CSV principal gerado pelo último código.
ARQ_METRICAS_RF = os.path.join(
    PASTA_RESULTADOS_RF,
    "metricas_amostra_a_amostra_rf_todas_larguras.csv"
)

# Saída desta V2.
PASTA_EXTRA = os.path.join(PASTA_RESULTADOS_RF, "graficos_pesquisa_extras_V2")
PASTA_HEATMAPS = os.path.join(PASTA_EXTRA, "01_heatmaps_limpos")
PASTA_SEPARACAO = os.path.join(PASTA_EXTRA, "02_separacao_e_contraste")
PASTA_RANKING = os.path.join(PASTA_EXTRA, "03_ranking_e_score")
PASTA_ROBUSTEZ = os.path.join(PASTA_EXTRA, "04_robustez_termica")
PASTA_PARETO = os.path.join(PASTA_EXTRA, "05_pareto")
PASTA_PARK = os.path.join(PASTA_EXTRA, "06_RF_vs_Park")
PASTA_CURVAS = os.path.join(PASTA_EXTRA, "07_curvas_exemplo")

for _p in [
    PASTA_EXTRA, PASTA_HEATMAPS, PASTA_SEPARACAO, PASTA_RANKING,
    PASTA_ROBUSTEZ, PASTA_PARETO, PASTA_PARK, PASTA_CURVAS
]:
    os.makedirs(_p, exist_ok=True)

# Faixas acumuladas esperadas: 30–40, 30–50, ..., 30–100 kHz.
FREQ_INICIO_FIXO_KHZ = globals().get("FREQ_INICIO_FIXO_KHZ", 30)
FREQ_MAXIMOS_KHZ = globals().get("FREQ_MAXIMOS_KHZ", list(range(40, 101, 10)))
FAIXAS_ACUMULADAS = [(float(FREQ_INICIO_FIXO_KHZ), float(fmax)) for fmax in FREQ_MAXIMOS_KHZ]

# Temperaturas do gráfico. Se não tiver exatamente, pega a mais próxima.
TEMPERATURAS_ALVO = list(range(-10, 81, 10))

DANOS_PLOTAR = [0, 1, 2]
METRICAS_PLOTAR = ["RMSD", "CCDM"]
SALVAR_PDF = True

# Controle de poluição visual.
MOSTRAR_VALORES_HEATMAP = True
MAX_CELULAS_COM_TEXTO = 90     # acima disso, tira texto das células para não ficar feio
FONTE_CELULA = 9
FONTE_TICKS_X = 17
FONTE_TICKS_Y = 18

# Park opcional. Se ficar pesado, coloque False.
GERAR_COMPARACAO_PARK = True
ARQ_METRICAS_PARK = os.path.join(PASTA_PARK, "metricas_amostra_a_amostra_park_todas_larguras.csv")

# Parâmetros Park iguais ao código original.
PARK_MAX_SHIFT_FRAC = globals().get("PARK_MAX_SHIFT_FRAC", 0.25)
PARK_SMOOTH_WIN = globals().get("PARK_SMOOTH_WIN", 5)
PARK_NSTEPS = globals().get("PARK_NSTEPS", 151)

# Parâmetros RF, usados só para curvas exemplo se precisar recomputar.
RF_COMP_POINT_PARAMS = globals().get("RF_COMP_POINT_PARAMS", dict(
    n_estimators=250,
    max_depth=10,
    min_samples_leaf=2,
    min_samples_split=4,
    max_features="sqrt",
    n_jobs=-1,
    random_state=0,
))

# Curvas exemplo.
GERAR_CURVAS_EXEMPLO = True
TEMPS_CURVAS_EXEMPLO = [-10, 30, 80]
FAIXA_CURVAS_EXEMPLO = None  # exemplo: (30, 70). Se None, escolhe automaticamente pelo score.


# ============================================================
# 3) ESTILO VISUAL — BONITO E MENOS SOBREPOSTO
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 18,
    "axes.labelsize": 23,
    "axes.titlesize": 24,
    "xtick.labelsize": FONTE_TICKS_X,
    "ytick.labelsize": FONTE_TICKS_Y,
    "legend.fontsize": 15,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

CORES_DANO = {
    0: "tab:blue",
    1: "tab:orange",
    2: "tab:red",
}

CORES_METODO = {
    "RF_ponto_a_ponto": "tab:blue",
    "Park": "tab:green",
}

NOME_METODO = {
    "RF_ponto_a_ponto": "RF ponto a ponto",
    "Park": "Park",
}


# ============================================================
# 4) FUNÇÕES BÁSICAS
# ============================================================

def salvar_fig(fig, pasta, nome_base):
    os.makedirs(pasta, exist_ok=True)
    png_path = os.path.join(pasta, nome_base + ".png")
    fig.savefig(png_path, dpi=600, bbox_inches="tight")

    if SALVAR_PDF:
        pdf_path = os.path.join(pasta, nome_base + ".pdf")
        fig.savefig(pdf_path, bbox_inches="tight")
        print(f"✅ Salvo:\n{png_path}\n{pdf_path}")
    else:
        print(f"✅ Salvo:\n{png_path}")


def estilo_eixos(ax):
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", labelsize=18)


def format_temp_num(T):
    T = float(T)
    if np.isclose(T, round(T)):
        return f"{int(round(T))}"
    return f"{T:.1f}"


def format_temp_label(T_alvo, T_usada=None):
    if T_usada is None or np.isclose(float(T_alvo), float(T_usada), atol=1e-9):
        return f"{format_temp_num(T_alvo)}°C"
    return f"{format_temp_num(T_alvo)}°C\nuso {format_temp_num(T_usada)}°C"


def zscore_col(s):
    s = pd.Series(s, dtype=float)
    std = s.std()
    if std == 0 or np.isnan(std):
        return s * 0.0
    return (s - s.mean()) / std


def safe_div(a, b, eps=1e-18):
    return np.asarray(a, dtype=float) / (np.abs(np.asarray(b, dtype=float)) + eps)


# ============================================================
# 5) FALLBACKS PARA RODAR SEPARADO
# ============================================================

if "extract_freq_hz" not in globals():
    def extract_freq_hz(col):
        m = re.match(r"^f_(\d+(?:\.\d+)?)Hz$", str(col))
        return float(m.group(1)) if m else None


if "get_freq_columns" not in globals():
    def get_freq_columns(df, fmin_khz, fmax_khz):
        cols, freqs = [], []
        for c in df.columns:
            f = extract_freq_hz(c)
            if f is not None:
                f_khz = f / 1e3
                if fmin_khz <= f_khz <= fmax_khz:
                    cols.append(c)
                    freqs.append(f)
        if len(cols) == 0:
            return [], np.array([])
        order = np.argsort(freqs)
        cols = [cols[i] for i in order]
        freqs = np.array(freqs)[order]
        return cols, freqs


if "moving_average" not in globals():
    def moving_average(arr, win):
        arr = np.asarray(arr, dtype=float)
        if win <= 1 or win % 2 == 0:
            return arr.copy()
        pad = win // 2
        arr_pad = np.pad(arr, (pad, pad), mode="edge")
        kernel = np.ones(win) / win
        smooth = np.convolve(arr_pad, kernel, mode="valid")
        if len(smooth) > len(arr):
            smooth = smooth[:len(arr)]
        elif len(smooth) < len(arr):
            smooth = np.pad(smooth, (0, len(arr) - len(smooth)), mode="edge")
        return smooth


if "add_extra_features_matrix" not in globals():
    def add_extra_features_matrix(X):
        mu = X.mean(axis=1, keepdims=True)
        sd = X.std(axis=1, keepdims=True)
        amp = (X.max(axis=1) - X.min(axis=1)).reshape(-1, 1)
        return np.hstack([X, mu, sd, amp])


if "rmsd" not in globals():
    def rmsd(y, ref):
        y = np.asarray(y, dtype=float)
        ref = np.asarray(ref, dtype=float)
        return float(np.sqrt(np.mean((y - ref) ** 2)))


if "ccdm" not in globals():
    def ccdm(y, ref):
        y = np.asarray(y, dtype=float)
        ref = np.asarray(ref, dtype=float)
        y0 = y - np.mean(y)
        r0 = ref - np.mean(ref)
        den = float(np.sqrt(np.sum(y0 ** 2) * np.sum(r0 ** 2))) + 1e-18
        corr = float(np.sum(y0 * r0)) / den
        return float(1 - corr)


if "curva_referencia_saudavel" not in globals():
    def curva_referencia_saudavel(df, fcols):
        df_sem = df[df["falha"] == 0]
        pool = df_sem.loc[np.isclose(df_sem["temperatura_c"], REF_TEMP), fcols].to_numpy(float)
        if len(pool) > 0:
            return np.median(pool, axis=0)
        print(f"⚠️ Não achei sem dano em {REF_TEMP}°C. Usando mediana geral sem dano.")
        return np.median(df_sem[fcols].to_numpy(float), axis=0)


if "compensar_rf_direto" not in globals():
    def compensar_rf_direto(df, fcols):
        df_sem = df[df["falha"] == 0]
        X_sem = df_sem[fcols].to_numpy(float)
        T_sem = df_sem["temperatura_c"].to_numpy(float)
        y_ref = curva_referencia_saudavel(df, fcols)

        Y_target = y_ref[None, :] - X_sem
        X_in = np.hstack([add_extra_features_matrix(X_sem), T_sem.reshape(-1, 1)])

        rf = RandomForestRegressor(**RF_COMP_POINT_PARAMS)
        rf.fit(X_in, Y_target)

        X_all = df[fcols].to_numpy(float)
        T_all = df["temperatura_c"].to_numpy(float)
        X_all_in = np.hstack([add_extra_features_matrix(X_all), T_all.reshape(-1, 1)])

        Y_comp = X_all + rf.predict(X_all_in)
        for i in range(len(Y_comp)):
            Y_comp[i] = moving_average(Y_comp[i], SMOOTH_WIN)

        df2 = df.copy()
        df2[fcols] = Y_comp
        return df2, y_ref


# ============================================================
# 6) CARREGAMENTO E PADRONIZAÇÃO DAS MÉTRICAS
# ============================================================

def procurar_csv_metricas_rf():
    if os.path.exists(ARQ_METRICAS_RF):
        return ARQ_METRICAS_RF

    candidatos = []
    for root, _, files in os.walk(PASTA_RESULTADOS_RF):
        for name in files:
            n = name.lower()
            if n.endswith(".csv") and "metricas" in n and "rf" in n and ("largura" in n or "faixa" in n):
                candidatos.append(os.path.join(root, name))

    if len(candidatos) == 0:
        raise FileNotFoundError(
            f"Não encontrei o CSV de métricas RF. Procurei primeiro em:\n{ARQ_METRICAS_RF}\n"
            f"e depois dentro de:\n{PASTA_RESULTADOS_RF}"
        )

    candidatos = sorted(candidatos, key=lambda p: os.path.getmtime(p), reverse=True)
    print(f"⚠️ CSV padrão não encontrado. Usando o mais recente encontrado:\n{candidatos[0]}")
    return candidatos[0]


def preparar_metricas(df):
    df = df.copy()

    # Garante nomes fundamentais.
    ren = {}
    for c in df.columns:
        cl = str(c).strip().lower()
        if cl in ["temp", "temperatura", "temperature", "temperatura_c"]:
            ren[c] = "temperatura_c"
        elif cl in ["dano", "damage", "falha", "estado"]:
            ren[c] = "falha"
        elif cl in ["rmsd", "rmsd_medio"] and "RMSD" not in df.columns:
            ren[c] = "RMSD"
        elif cl in ["ccdm", "ccdm_medio"] and "CCDM" not in df.columns:
            ren[c] = "CCDM"
    df = df.rename(columns=ren)

    required = ["temperatura_c", "falha", "RMSD", "CCDM", "faixa_min_khz", "faixa_max_khz"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Métricas sem colunas obrigatórias: {missing}")

    df["temperatura_c"] = pd.to_numeric(df["temperatura_c"], errors="coerce")
    df["falha"] = pd.to_numeric(df["falha"], errors="coerce").astype(int)
    df["RMSD"] = pd.to_numeric(df["RMSD"], errors="coerce")
    df["CCDM"] = pd.to_numeric(df["CCDM"], errors="coerce")
    df["faixa_min_khz"] = pd.to_numeric(df["faixa_min_khz"], errors="coerce")
    df["faixa_max_khz"] = pd.to_numeric(df["faixa_max_khz"], errors="coerce")
    df["largura_khz"] = df["faixa_max_khz"] - df["faixa_min_khz"]

    if "faixa_label" not in df.columns:
        df["faixa_label"] = (
            df["faixa_min_khz"].round().astype(int).astype(str)
            + "–" +
            df["faixa_max_khz"].round().astype(int).astype(str)
            + " kHz"
        )

    if "metodo" not in df.columns:
        df["metodo"] = "RF_ponto_a_ponto"

    # Remove linhas inválidas.
    df = df.dropna(subset=["temperatura_c", "falha", "RMSD", "CCDM", "faixa_min_khz", "faixa_max_khz"])

    # Mantém só as faixas acumuladas configuradas.
    faixas_set = set((float(a), float(b)) for a, b in FAIXAS_ACUMULADAS)
    df = df[
        df.apply(lambda r: (float(r["faixa_min_khz"]), float(r["faixa_max_khz"])) in faixas_set, axis=1)
    ].copy()

    return df.sort_values([
        "metodo", "faixa_min_khz", "faixa_max_khz", "temperatura_c", "falha"
    ]).reset_index(drop=True)


def carregar_metricas_rf():
    if "df_metricas_rf" in globals():
        print("✅ Usando df_metricas_rf já existente na memória.")
        return preparar_metricas(globals()["df_metricas_rf"])

    path = procurar_csv_metricas_rf()
    print(f"🔹 Carregando métricas RF de:\n{path}")
    return preparar_metricas(pd.read_csv(path))


def carregar_base_extra():
    if "df_base" in globals():
        print("✅ Usando df_base já existente na memória.")
        return globals()["df_base"].copy()

    if not os.path.exists(ARQ_BASE):
        raise FileNotFoundError(f"Não encontrei {ARQ_BASE}. Necessário para Park e curvas exemplo.")

    print(f"🔹 Carregando base de:\n{ARQ_BASE}")
    df = pd.read_pickle(ARQ_BASE)
    df["temperatura_c"] = pd.to_numeric(df["temperatura_c"], errors="coerce")
    df["falha"] = pd.to_numeric(df["falha"], errors="coerce").astype(int)
    return df


def ordem_faixas(df):
    return (
        df[["faixa_min_khz", "faixa_max_khz", "faixa_label"]]
        .drop_duplicates()
        .sort_values(["faixa_min_khz", "faixa_max_khz"])
        .reset_index(drop=True)
    )


# ============================================================
# 7) TEMPERATURAS ALVO COM MAIS PRÓXIMA DISPONÍVEL
# ============================================================

def montar_mapa_temperaturas(df_metricas, metodo="RF_ponto_a_ponto"):
    """
    Para cada temperatura alvo em TEMPERATURAS_ALVO, escolhe a temperatura real
    mais próxima que tenha pelo menos os danos 0,1,2 para o método.
    """
    sub = df_metricas[df_metricas["metodo"] == metodo].copy()
    if len(sub) == 0:
        return pd.DataFrame()

    temps_reais = sorted(sub["temperatura_c"].dropna().unique())
    rows = []

    for T_alvo in TEMPERATURAS_ALVO:
        candidatos = []
        for T in temps_reais:
            ok_danos = True
            for d in DANOS_PLOTAR:
                if not np.any(np.isclose(sub["temperatura_c"], T) & (sub["falha"] == d)):
                    ok_danos = False
                    break
            if ok_danos:
                candidatos.append(float(T))

        if len(candidatos) == 0:
            continue

        candidatos = np.asarray(candidatos, dtype=float)
        T_usada = float(candidatos[np.argmin(np.abs(candidatos - float(T_alvo)))])
        rows.append({
            "metodo": metodo,
            "temp_alvo_c": float(T_alvo),
            "temp_usada_c": T_usada,
            "erro_temp_c": abs(T_usada - float(T_alvo)),
            "temp_label": format_temp_label(T_alvo, T_usada),
        })

    mapa = pd.DataFrame(rows)
    return mapa


def aplicar_temperaturas_alvo(df_metricas):
    """Duplica/filtra as métricas para as temperaturas alvo, usando a real mais próxima."""
    partes = []
    mapas = []

    for metodo in sorted(df_metricas["metodo"].unique()):
        mapa = montar_mapa_temperaturas(df_metricas, metodo=metodo)
        if len(mapa) == 0:
            continue
        mapas.append(mapa)
        subm = df_metricas[df_metricas["metodo"] == metodo].copy()

        for _, r in mapa.iterrows():
            sub = subm[np.isclose(subm["temperatura_c"], r["temp_usada_c"])].copy()
            sub["temp_alvo_c"] = float(r["temp_alvo_c"])
            sub["temp_usada_c"] = float(r["temp_usada_c"])
            sub["erro_temp_c"] = float(r["erro_temp_c"])
            sub["temp_label"] = r["temp_label"]
            partes.append(sub)

    if len(partes) == 0:
        raise RuntimeError("Nenhuma temperatura alvo pôde ser montada.")

    df_plot = pd.concat(partes, ignore_index=True)
    mapa_total = pd.concat(mapas, ignore_index=True) if len(mapas) else pd.DataFrame()

    path_mapa = os.path.join(PASTA_EXTRA, "mapa_temperaturas_alvo_para_temperaturas_reais.csv")
    mapa_total.to_csv(path_mapa, index=False)
    print(f"✅ Mapa de temperaturas salvo em: {path_mapa}")

    return df_plot, mapa_total


def resumo_metricas(df_metricas):
    group_cols = [
        "metodo", "faixa_min_khz", "faixa_max_khz", "largura_khz", "faixa_label",
        "temp_alvo_c", "temp_usada_c", "temp_label", "falha"
    ]
    return (
        df_metricas
        .groupby(group_cols, as_index=False)
        .agg(
            RMSD_medio=("RMSD", "mean"),
            RMSD_std=("RMSD", "std"),
            CCDM_medio=("CCDM", "mean"),
            CCDM_std=("CCDM", "std"),
            n_amostras=("RMSD", "size"),
        )
    )


# ============================================================
# 8) HEATMAP LIMPO — MÉTRICA POR DANO
# ============================================================

def matriz_pivot(df_sum, metodo, dano, metrica):
    col = f"{metrica}_medio"
    sub = df_sum[(df_sum["metodo"] == metodo) & (df_sum["falha"] == dano)].copy()
    faixas = ordem_faixas(sub)
    labels_faixa = faixas["faixa_label"].tolist()
    temps = sorted(sub["temp_alvo_c"].dropna().unique())
    labels_temp = []
    for T in temps:
        ss = sub[np.isclose(sub["temp_alvo_c"], T)]
        if len(ss):
            labels_temp.append(str(ss["temp_label"].iloc[0]))
        else:
            labels_temp.append(format_temp_label(T))

    pivot = sub.pivot_table(index="temp_alvo_c", columns="faixa_label", values=col, aggfunc="mean")
    pivot = pivot.reindex(index=temps, columns=labels_faixa)
    return pivot, labels_faixa, labels_temp


def desenhar_heatmap(ax, pivot, labels_x, labels_y, titulo, cbar_label=None, cmap="viridis", vmin=None, vmax=None, texto=True):
    im = ax.imshow(pivot.values, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(titulo, fontsize=24, pad=14)
    ax.set_xticks(np.arange(len(labels_x)))
    ax.set_xticklabels(labels_x, rotation=35, ha="right", fontsize=FONTE_TICKS_X)
    ax.set_yticks(np.arange(len(labels_y)))
    ax.set_yticklabels(labels_y, fontsize=FONTE_TICKS_Y)
    ax.set_xlabel("Faixa analisada", fontsize=23, labelpad=10)
    estilo_eixos(ax)

    ncel = pivot.shape[0] * pivot.shape[1]
    if texto and MOSTRAR_VALORES_HEATMAP and ncel <= MAX_CELULAS_COM_TEXTO:
        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = pivot.values[i, j]
                if np.isfinite(val):
                    ax.text(j, i, f"{val:.2g}", ha="center", va="center", fontsize=FONTE_CELULA, color="black")

    return im


def plot_heatmap_metrica_individual(df_sum, metrica="RMSD", metodo="RF_ponto_a_ponto"):
    for dano in DANOS_PLOTAR:
        pivot, labels_x, labels_y = matriz_pivot(df_sum, metodo, dano, metrica)
        if pivot.empty:
            continue

        fig_w = max(11, 1.15 * len(labels_x) + 5)
        fig_h = max(7, 0.52 * len(labels_y) + 3.5)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=300)

        im = desenhar_heatmap(
            ax, pivot, labels_x, labels_y,
            titulo=f"{NOME_METODO.get(metodo, metodo)} — {metrica} — dano {dano}",
            cmap="viridis",
        )
        ax.set_ylabel("Temperatura", fontsize=23, labelpad=10)
        cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
        cbar.set_label(metrica, fontsize=21)
        cbar.ax.tick_params(labelsize=16)

        fig.tight_layout()
        salvar_fig(fig, PASTA_HEATMAPS, f"heatmap_limpo_{metodo}_{metrica}_dano{dano}")
        plt.show()


def plot_heatmap_metrica_triptico(df_sum, metrica="RMSD", metodo="RF_ponto_a_ponto"):
    pivots = []
    labels_x_ref, labels_y_ref = None, None
    vals = []

    for dano in DANOS_PLOTAR:
        pivot, labels_x, labels_y = matriz_pivot(df_sum, metodo, dano, metrica)
        pivots.append((dano, pivot, labels_x, labels_y))
        labels_x_ref = labels_x
        labels_y_ref = labels_y
        vals.append(pivot.values)

    finite = np.concatenate([v[np.isfinite(v)] for v in vals if np.any(np.isfinite(v))])
    if len(finite) == 0:
        return None

    vmin = np.nanmin(finite)
    vmax = np.nanmax(finite)

    fig_w = max(22, 6.8 * len(DANOS_PLOTAR))
    fig_h = max(7.5, 0.55 * len(labels_y_ref) + 3.5)
    fig, axes = plt.subplots(1, len(DANOS_PLOTAR), figsize=(fig_w, fig_h), dpi=300, sharey=True)

    if len(DANOS_PLOTAR) == 1:
        axes = [axes]

    im_last = None
    for ax, (dano, pivot, labels_x, labels_y) in zip(axes, pivots):
        im_last = desenhar_heatmap(
            ax, pivot, labels_x, labels_y,
            titulo=f"Dano {dano}",
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
            texto=True,
        )

    axes[0].set_ylabel("Temperatura", fontsize=23, labelpad=10)
    cbar = fig.colorbar(im_last, ax=axes, fraction=0.022, pad=0.018)
    cbar.set_label(metrica, fontsize=21)
    cbar.ax.tick_params(labelsize=16)

    fig.suptitle(
        f"{NOME_METODO.get(metodo, metodo)} — {metrica}: temperatura × largura da faixa — referência {REF_TEMP}°C",
        fontsize=28,
        y=1.02,
    )
    fig.tight_layout(rect=[0, 0, 0.965, 0.95])
    salvar_fig(fig, PASTA_HEATMAPS, f"heatmap_triptico_{metodo}_{metrica}")
    plt.show()
    return fig


# ============================================================
# 9) SEPARAÇÃO, CONTRASTE E INVERSÕES
# ============================================================

def calcular_tabela_separacao(df_sum):
    rows = []
    group_cols = [
        "metodo", "faixa_min_khz", "faixa_max_khz", "largura_khz", "faixa_label",
        "temp_alvo_c", "temp_usada_c", "temp_label"
    ]

    for keys, g in df_sum.groupby(group_cols):
        metodo, fmin, fmax, largura, faixa_label, T_alvo, T_usada, T_label = keys
        gd = g.set_index("falha")
        if not all(d in gd.index for d in [0, 1, 2]):
            continue

        for metrica in METRICAS_PLOTAR:
            v0 = float(gd.loc[0, f"{metrica}_medio"])
            v1 = float(gd.loc[1, f"{metrica}_medio"])
            v2 = float(gd.loc[2, f"{metrica}_medio"])

            rows.append({
                "metodo": metodo,
                "faixa_min_khz": fmin,
                "faixa_max_khz": fmax,
                "largura_khz": largura,
                "faixa_label": faixa_label,
                "temp_alvo_c": T_alvo,
                "temp_usada_c": T_usada,
                "temp_label": T_label,
                "metrica": metrica,
                "D0": v0,
                "D1": v1,
                "D2": v2,
                "sep_D1_D0": v1 - v0,
                "sep_D2_D1": v2 - v1,
                "sep_D2_D0": v2 - v0,
                "ratio_D1_D0": (v1 + 1e-18) / (abs(v0) + 1e-18),
                "ratio_D2_D0": (v2 + 1e-18) / (abs(v0) + 1e-18),
                "inv_D1_D0": int((v1 - v0) <= 0),
                "inv_D2_D1": int((v2 - v1) <= 0),
            })

    return pd.DataFrame(rows)


def plot_heatmap_separacao_individual(df_sep, metrica="RMSD", metodo="RF_ponto_a_ponto"):
    sub = df_sep[(df_sep["metodo"] == metodo) & (df_sep["metrica"] == metrica)].copy()
    if len(sub) == 0:
        return

    variaveis = ["sep_D1_D0", "sep_D2_D1", "sep_D2_D0"]
    titulos = {
        "sep_D1_D0": "Separação D1 − D0",
        "sep_D2_D1": "Separação D2 − D1",
        "sep_D2_D0": "Separação D2 − D0",
    }

    faixas = ordem_faixas(sub)
    labels_x = faixas["faixa_label"].tolist()
    temps = sorted(sub["temp_alvo_c"].dropna().unique())
    labels_y = []
    for T in temps:
        ss = sub[np.isclose(sub["temp_alvo_c"], T)]
        labels_y.append(str(ss["temp_label"].iloc[0]) if len(ss) else format_temp_label(T))

    for var in variaveis:
        pivot = sub.pivot_table(index="temp_alvo_c", columns="faixa_label", values=var, aggfunc="mean")
        pivot = pivot.reindex(index=temps, columns=labels_x)
        finite = pivot.values[np.isfinite(pivot.values)]
        vmax_abs = np.nanmax(np.abs(finite)) if len(finite) else 1.0

        fig_w = max(11, 1.15 * len(labels_x) + 5)
        fig_h = max(7, 0.52 * len(labels_y) + 3.5)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=300)

        im = desenhar_heatmap(
            ax, pivot, labels_x, labels_y,
            titulo=f"{NOME_METODO.get(metodo, metodo)} — {titulos[var]} em {metrica}",
            cmap="coolwarm",
            vmin=-vmax_abs,
            vmax=vmax_abs,
        )
        ax.set_ylabel("Temperatura", fontsize=23, labelpad=10)
        ax.axhline(-0.5, color="none")
        cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
        cbar.set_label(f"Separação em {metrica}", fontsize=21)
        cbar.ax.tick_params(labelsize=16)

        fig.tight_layout()
        salvar_fig(fig, PASTA_SEPARACAO, f"heatmap_{metodo}_{metrica}_{var}")
        plt.show()


def plot_heatmap_contraste_relativo(df_sep, metrica="RMSD", metodo="RF_ponto_a_ponto"):
    sub = df_sep[(df_sep["metodo"] == metodo) & (df_sep["metrica"] == metrica)].copy()
    if len(sub) == 0:
        return

    variaveis = ["ratio_D1_D0", "ratio_D2_D0"]
    titulos = {
        "ratio_D1_D0": "Contraste relativo D1 / D0",
        "ratio_D2_D0": "Contraste relativo D2 / D0",
    }

    faixas = ordem_faixas(sub)
    labels_x = faixas["faixa_label"].tolist()
    temps = sorted(sub["temp_alvo_c"].dropna().unique())
    labels_y = []
    for T in temps:
        ss = sub[np.isclose(sub["temp_alvo_c"], T)]
        labels_y.append(str(ss["temp_label"].iloc[0]) if len(ss) else format_temp_label(T))

    fig, axes = plt.subplots(1, 2, figsize=(20, max(7, 0.55 * len(labels_y) + 3)), dpi=300, sharey=True)

    im_last = None
    for ax, var in zip(axes, variaveis):
        pivot = sub.pivot_table(index="temp_alvo_c", columns="faixa_label", values=var, aggfunc="mean")
        pivot = pivot.reindex(index=temps, columns=labels_x)
        im_last = desenhar_heatmap(
            ax, pivot, labels_x, labels_y,
            titulo=titulos[var],
            cmap="viridis",
            texto=True,
        )

    axes[0].set_ylabel("Temperatura", fontsize=23, labelpad=10)
    cbar = fig.colorbar(im_last, ax=axes, fraction=0.025, pad=0.018)
    cbar.set_label("Razão", fontsize=21)
    cbar.ax.tick_params(labelsize=16)
    fig.suptitle(
        f"{NOME_METODO.get(metodo, metodo)} — contraste relativo em {metrica}",
        fontsize=28,
        y=1.02,
    )
    fig.tight_layout(rect=[0, 0, 0.965, 0.95])
    salvar_fig(fig, PASTA_SEPARACAO, f"heatmap_contraste_relativo_{metodo}_{metrica}")
    plt.show()


def plot_mapa_inversoes(df_sep, metodo="RF_ponto_a_ponto"):
    sub = df_sep[df_sep["metodo"] == metodo].copy()
    if len(sub) == 0:
        return

    inv = (
        sub.groupby([
            "faixa_min_khz", "faixa_max_khz", "largura_khz", "faixa_label",
            "temp_alvo_c", "temp_label"
        ], as_index=False)
        .agg(
            inversoes_total=("inv_D1_D0", "sum"),
            inversoes_D2_D1=("inv_D2_D1", "sum"),
        )
    )
    # inversoes_total soma inv_D1_D0 nas duas métricas; agora adiciona D2_D1.
    inv2 = (
        sub.groupby([
            "faixa_min_khz", "faixa_max_khz", "largura_khz", "faixa_label",
            "temp_alvo_c", "temp_label"
        ], as_index=False)
        .agg(total_D2_D1=("inv_D2_D1", "sum"))
    )
    inv = inv.merge(inv2, on=["faixa_min_khz", "faixa_max_khz", "largura_khz", "faixa_label", "temp_alvo_c", "temp_label"])
    inv["inversoes_total"] = inv["inversoes_total"] + inv["total_D2_D1"]

    faixas = ordem_faixas(inv)
    labels_x = faixas["faixa_label"].tolist()
    temps = sorted(inv["temp_alvo_c"].dropna().unique())
    labels_y = []
    for T in temps:
        ss = inv[np.isclose(inv["temp_alvo_c"], T)]
        labels_y.append(str(ss["temp_label"].iloc[0]) if len(ss) else format_temp_label(T))

    pivot = inv.pivot_table(index="temp_alvo_c", columns="faixa_label", values="inversoes_total", aggfunc="mean")
    pivot = pivot.reindex(index=temps, columns=labels_x)

    fig, ax = plt.subplots(figsize=(max(11, 1.15 * len(labels_x) + 5), max(7, 0.52 * len(labels_y) + 3.5)), dpi=300)
    im = desenhar_heatmap(
        ax, pivot, labels_x, labels_y,
        titulo=f"{NOME_METODO.get(metodo, metodo)} — inversões da ordem esperada D0 < D1 < D2",
        cmap="magma_r",
        vmin=0,
        vmax=4,
        texto=True,
    )
    ax.set_ylabel("Temperatura", fontsize=23, labelpad=10)
    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
    cbar.set_label("Inversões em RMSD/CCDM", fontsize=21)
    cbar.ax.tick_params(labelsize=16)
    fig.tight_layout()
    salvar_fig(fig, PASTA_SEPARACAO, f"mapa_inversoes_{metodo}")
    plt.show()


# ============================================================
# 10) SCORE DE PESQUISA E RANKING
# ============================================================

def montar_score_pesquisa(df_sep):
    rows = []
    idx_cols = [
        "metodo", "faixa_min_khz", "faixa_max_khz", "largura_khz", "faixa_label",
        "temp_alvo_c", "temp_usada_c", "temp_label"
    ]

    for keys, g in df_sep.groupby(idx_cols):
        metodo, fmin, fmax, largura, faixa_label, T_alvo, T_usada, T_label = keys
        gm = g.set_index("metrica")
        if not all(m in gm.index for m in ["RMSD", "CCDM"]):
            continue

        r = gm.loc["RMSD"]
        c = gm.loc["CCDM"]

        inversoes = int(r["inv_D1_D0"] + r["inv_D2_D1"] + c["inv_D1_D0"] + c["inv_D2_D1"])

        rows.append({
            "metodo": metodo,
            "faixa_min_khz": fmin,
            "faixa_max_khz": fmax,
            "largura_khz": largura,
            "faixa_label": faixa_label,
            "temp_alvo_c": T_alvo,
            "temp_usada_c": T_usada,
            "temp_label": T_label,
            "deltaT_abs_c": abs(float(T_alvo) - float(REF_TEMP)),
            "RMSD_D0": float(r["D0"]),
            "RMSD_D1": float(r["D1"]),
            "RMSD_D2": float(r["D2"]),
            "CCDM_D0": float(c["D0"]),
            "CCDM_D1": float(c["D1"]),
            "CCDM_D2": float(c["D2"]),
            "sep_RMSD_D1_D0": float(r["sep_D1_D0"]),
            "sep_RMSD_D2_D1": float(r["sep_D2_D1"]),
            "sep_RMSD_D2_D0": float(r["sep_D2_D0"]),
            "sep_CCDM_D1_D0": float(c["sep_D1_D0"]),
            "sep_CCDM_D2_D1": float(c["sep_D2_D1"]),
            "sep_CCDM_D2_D0": float(c["sep_D2_D0"]),
            "ratio_RMSD_D2_D0": float(r["ratio_D2_D0"]),
            "ratio_CCDM_D2_D0": float(c["ratio_D2_D0"]),
            "inversoes_total": inversoes,
        })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    partes = []
    for metodo, gm in df.groupby("metodo"):
        gm = gm.copy()

        # Score menor = melhor.
        # Penaliza erro do sem dano e inversões.
        # Premia separação/contraste entre danos.
        gm["score_pesquisa"] = (
            1.00 * zscore_col(gm["RMSD_D0"]) +
            1.00 * zscore_col(gm["CCDM_D0"]) -
            0.50 * zscore_col(gm["sep_RMSD_D1_D0"]) -
            0.50 * zscore_col(gm["sep_RMSD_D2_D1"]) -
            0.40 * zscore_col(gm["sep_RMSD_D2_D0"]) -
            0.50 * zscore_col(gm["sep_CCDM_D1_D0"]) -
            0.50 * zscore_col(gm["sep_CCDM_D2_D1"]) -
            0.40 * zscore_col(gm["sep_CCDM_D2_D0"]) +
            1.60 * gm["inversoes_total"]
        )
        partes.append(gm)

    return pd.concat(partes, ignore_index=True).sort_values(["metodo", "score_pesquisa"]).reset_index(drop=True)


def plot_score_heatmap(df_score, metodo="RF_ponto_a_ponto"):
    sub = df_score[df_score["metodo"] == metodo].copy()
    if len(sub) == 0:
        return

    faixas = ordem_faixas(sub)
    labels_x = faixas["faixa_label"].tolist()
    temps = sorted(sub["temp_alvo_c"].dropna().unique())
    labels_y = []
    for T in temps:
        ss = sub[np.isclose(sub["temp_alvo_c"], T)]
        labels_y.append(str(ss["temp_label"].iloc[0]) if len(ss) else format_temp_label(T))

    pivot = sub.pivot_table(index="temp_alvo_c", columns="faixa_label", values="score_pesquisa", aggfunc="mean")
    pivot = pivot.reindex(index=temps, columns=labels_x)

    fig, ax = plt.subplots(figsize=(max(11, 1.15 * len(labels_x) + 5), max(7, 0.52 * len(labels_y) + 3.5)), dpi=300)
    im = desenhar_heatmap(
        ax, pivot, labels_x, labels_y,
        titulo=f"{NOME_METODO.get(metodo, metodo)} — score de escolha da faixa\nmenor = melhor",
        cmap="viridis_r",
        texto=True,
    )
    ax.set_ylabel("Temperatura", fontsize=23, labelpad=10)
    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
    cbar.set_label("Score", fontsize=21)
    cbar.ax.tick_params(labelsize=16)
    fig.tight_layout()
    salvar_fig(fig, PASTA_RANKING, f"score_heatmap_{metodo}")
    plt.show()


def plot_ranking_medio_faixas(df_score, metodo="RF_ponto_a_ponto"):
    sub = df_score[df_score["metodo"] == metodo].copy()
    if len(sub) == 0:
        return None

    rank = (
        sub.groupby(["faixa_min_khz", "faixa_max_khz", "largura_khz", "faixa_label"], as_index=False)
        .agg(
            score_medio=("score_pesquisa", "mean"),
            score_std=("score_pesquisa", "std"),
            inversoes_media=("inversoes_total", "mean"),
            RMSD_D0_medio=("RMSD_D0", "mean"),
            CCDM_D0_medio=("CCDM_D0", "mean"),
            sep_RMSD_D2_D0_medio=("sep_RMSD_D2_D0", "mean"),
            sep_CCDM_D2_D0_medio=("sep_CCDM_D2_D0", "mean"),
        )
        .sort_values("score_medio", ascending=True)
        .reset_index(drop=True)
    )

    csv_path = os.path.join(PASTA_RANKING, f"ranking_medio_faixas_{metodo}.csv")
    rank.to_csv(csv_path, index=False)
    print(f"✅ Ranking salvo em: {csv_path}")

    fig, ax = plt.subplots(figsize=(15, 8), dpi=300)
    x = np.arange(len(rank))
    ax.bar(
        x,
        rank["score_medio"],
        yerr=rank["score_std"].fillna(0),
        capsize=4,
        edgecolor="black",
        linewidth=0.9,
    )
    ax.set_xticks(x)
    ax.set_xticklabels(rank["faixa_label"], rotation=35, ha="right", fontsize=17)
    ax.set_ylabel("Score médio menor = melhor", fontsize=23)
    ax.set_xlabel("Faixa analisada", fontsize=23, labelpad=10)
    ax.set_title(f"{NOME_METODO.get(metodo, metodo)} — ranking médio das faixas", fontsize=25, pad=16)
    estilo_eixos(ax)
    fig.tight_layout()
    salvar_fig(fig, PASTA_RANKING, f"ranking_medio_faixas_{metodo}")
    plt.show()

    return rank


def plot_melhor_faixa_por_temperatura(df_score):
    if len(df_score) == 0:
        return

    fig, ax = plt.subplots(figsize=(15, 8), dpi=300)

    for metodo, gm in df_score.groupby("metodo"):
        best = (
            gm.sort_values("score_pesquisa", ascending=True)
            .groupby("temp_alvo_c", as_index=False)
            .first()
            .sort_values("temp_alvo_c")
        )

        ax.plot(
            best["temp_alvo_c"],
            best["faixa_max_khz"],
            marker="o",
            linewidth=2.8,
            markersize=8,
            label=NOME_METODO.get(metodo, metodo),
        )

        for _, r in best.iterrows():
            ax.text(
                r["temp_alvo_c"],
                r["faixa_max_khz"] + 1.0,
                str(r["faixa_label"]).replace(" kHz", ""),
                ha="center",
                va="bottom",
                fontsize=10,
            )

    ax.axvline(REF_TEMP, color="black", linestyle="--", linewidth=1.2, label=f"Referência {REF_TEMP}°C")
    ax.set_xlabel("Temperatura alvo (°C)", fontsize=23, labelpad=10)
    ax.set_ylabel("Frequência máxima da melhor faixa (kHz)", fontsize=23)
    ax.set_title("Melhor largura de faixa em função da temperatura", fontsize=25, pad=16)
    ax.legend(frameon=True, fontsize=15, loc="best")
    estilo_eixos(ax)
    fig.tight_layout()
    salvar_fig(fig, PASTA_RANKING, "melhor_faixa_por_temperatura")
    plt.show()


# ============================================================
# 11) ROBUSTEZ TÉRMICA E SENSIBILIDADE
# ============================================================

def plot_linhas_metricas_por_faixa(df_sum, metrica="RMSD", metodo="RF_ponto_a_ponto"):
    sub = df_sum[df_sum["metodo"] == metodo].copy()
    if len(sub) == 0:
        return

    col = f"{metrica}_medio"
    faixas = ordem_faixas(sub)

    fig, axes = plt.subplots(1, len(DANOS_PLOTAR), figsize=(7.5 * len(DANOS_PLOTAR), 7.5), dpi=300, sharey=True)
    if len(DANOS_PLOTAR) == 1:
        axes = [axes]

    for ax, dano in zip(axes, DANOS_PLOTAR):
        sd = sub[sub["falha"] == dano]
        for _, row in faixas.iterrows():
            sf = sd[
                np.isclose(sd["faixa_min_khz"], row["faixa_min_khz"]) &
                np.isclose(sd["faixa_max_khz"], row["faixa_max_khz"])
            ]
            g = sf.groupby("temp_alvo_c", as_index=False)[col].mean().sort_values("temp_alvo_c")
            ax.plot(g["temp_alvo_c"], g[col], marker="o", linewidth=2.0, markersize=6, label=row["faixa_label"])

        ax.axvline(REF_TEMP, color="black", linestyle="--", linewidth=1.0)
        ax.set_title(f"Dano {dano}", fontsize=24, pad=14)
        ax.set_xlabel("Temperatura alvo (°C)", fontsize=22, labelpad=10)
        estilo_eixos(ax)

    axes[0].set_ylabel(metrica, fontsize=23)
    axes[-1].legend(frameon=True, fontsize=11, loc="best")
    fig.suptitle(f"{NOME_METODO.get(metodo, metodo)} — {metrica} por temperatura e largura", fontsize=27, y=1.02)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    salvar_fig(fig, PASTA_ROBUSTEZ, f"linhas_{metodo}_{metrica}_por_temperatura_faixa")
    plt.show()


def calcular_sensibilidade_termica(df_sum):
    rows = []
    for (metodo, fmin, fmax, largura, faixa_label, dano), g in df_sum.groupby([
        "metodo", "faixa_min_khz", "faixa_max_khz", "largura_khz", "faixa_label", "falha"
    ]):
        g = g.copy()
        g["deltaT_abs_c"] = np.abs(g["temp_alvo_c"].astype(float) - float(REF_TEMP))
        for metrica in METRICAS_PLOTAR:
            col = f"{metrica}_medio"
            gg = g[["deltaT_abs_c", col]].dropna().groupby("deltaT_abs_c", as_index=False).mean()
            if len(gg) < 2:
                slope = np.nan
                intercept = np.nan
            else:
                slope, intercept = np.polyfit(gg["deltaT_abs_c"].to_numpy(float), gg[col].to_numpy(float), 1)

            rows.append({
                "metodo": metodo,
                "faixa_min_khz": fmin,
                "faixa_max_khz": fmax,
                "largura_khz": largura,
                "faixa_label": faixa_label,
                "falha": int(dano),
                "metrica": metrica,
                "slope_por_grau": slope,
                "intercept": intercept,
            })

    return pd.DataFrame(rows)


def plot_sensibilidade_termica(df_sens, metrica="RMSD", metodo="RF_ponto_a_ponto"):
    sub = df_sens[(df_sens["metodo"] == metodo) & (df_sens["metrica"] == metrica)].copy()
    if len(sub) == 0:
        return

    faixas = ordem_faixas(sub)
    labels_x = faixas["faixa_label"].tolist()

    fig, ax = plt.subplots(figsize=(15, 8), dpi=300)
    x = np.arange(len(labels_x))
    bar_w = 0.24

    for i, dano in enumerate(DANOS_PLOTAR):
        vals = []
        for _, row in faixas.iterrows():
            ss = sub[
                np.isclose(sub["faixa_min_khz"], row["faixa_min_khz"]) &
                np.isclose(sub["faixa_max_khz"], row["faixa_max_khz"]) &
                (sub["falha"] == dano)
            ]
            vals.append(float(ss["slope_por_grau"].iloc[0]) if len(ss) else np.nan)
        ax.bar(x + (i - 1) * bar_w, vals, width=bar_w, edgecolor="black", linewidth=0.8, label=f"Dano {dano}")

    ax.axhline(0, color="black", linewidth=1.0)
    ax.set_xticks(x)
    ax.set_xticklabels(labels_x, rotation=35, ha="right", fontsize=17)
    ax.set_ylabel(f"Inclinação de {metrica} por °C", fontsize=23)
    ax.set_xlabel("Faixa analisada", fontsize=23, labelpad=10)
    ax.set_title(f"{NOME_METODO.get(metodo, metodo)} — sensibilidade térmica de {metrica}", fontsize=25, pad=16)
    ax.legend(frameon=True, fontsize=15)
    estilo_eixos(ax)
    fig.tight_layout()
    salvar_fig(fig, PASTA_ROBUSTEZ, f"sensibilidade_termica_{metodo}_{metrica}")
    plt.show()


def plot_coef_variacao_por_faixa(df_sum, metrica="RMSD", metodo="RF_ponto_a_ponto"):
    sub = df_sum[df_sum["metodo"] == metodo].copy()
    if len(sub) == 0:
        return

    col = f"{metrica}_medio"
    rows = []
    for (fmin, fmax, faixa_label, dano), g in sub.groupby(["faixa_min_khz", "faixa_max_khz", "faixa_label", "falha"]):
        vals = g[col].to_numpy(float)
        cv = np.nanstd(vals) / (abs(np.nanmean(vals)) + 1e-18)
        rows.append({
            "faixa_min_khz": fmin,
            "faixa_max_khz": fmax,
            "faixa_label": faixa_label,
            "falha": int(dano),
            "cv": cv,
        })
    cvdf = pd.DataFrame(rows)

    faixas = ordem_faixas(cvdf)
    labels_x = faixas["faixa_label"].tolist()

    fig, ax = plt.subplots(figsize=(15, 8), dpi=300)
    x = np.arange(len(labels_x))
    bar_w = 0.24

    for i, dano in enumerate(DANOS_PLOTAR):
        vals = []
        for _, row in faixas.iterrows():
            ss = cvdf[
                np.isclose(cvdf["faixa_min_khz"], row["faixa_min_khz"]) &
                np.isclose(cvdf["faixa_max_khz"], row["faixa_max_khz"]) &
                (cvdf["falha"] == dano)
            ]
            vals.append(float(ss["cv"].iloc[0]) if len(ss) else np.nan)
        ax.bar(x + (i - 1) * bar_w, vals, width=bar_w, edgecolor="black", linewidth=0.8, label=f"Dano {dano}")

    ax.set_xticks(x)
    ax.set_xticklabels(labels_x, rotation=35, ha="right", fontsize=17)
    ax.set_ylabel(f"Coeficiente de variação de {metrica}", fontsize=23)
    ax.set_xlabel("Faixa analisada", fontsize=23, labelpad=10)
    ax.set_title(f"{NOME_METODO.get(metodo, metodo)} — estabilidade térmica de {metrica}", fontsize=25, pad=16)
    ax.legend(frameon=True, fontsize=15)
    estilo_eixos(ax)
    fig.tight_layout()
    salvar_fig(fig, PASTA_ROBUSTEZ, f"coef_variacao_{metodo}_{metrica}")
    plt.show()


# ============================================================
# 12) PARETO — ERRO NO SEM DANO × SEPARAÇÃO DE DANO
# ============================================================

def plot_pareto_erro_vs_separacao(df_score, metrica="RMSD", metodo="RF_ponto_a_ponto"):
    sub = df_score[df_score["metodo"] == metodo].copy()
    if len(sub) == 0:
        return

    if metrica == "RMSD":
        erro_col = "RMSD_D0"
        sep_col = "sep_RMSD_D2_D0"
    else:
        erro_col = "CCDM_D0"
        sep_col = "sep_CCDM_D2_D0"

    agg = (
        sub.groupby(["faixa_min_khz", "faixa_max_khz", "faixa_label"], as_index=False)
        .agg(
            erro_d0=(erro_col, "mean"),
            sep_d2_d0=(sep_col, "mean"),
            score=("score_pesquisa", "mean"),
        )
        .sort_values("faixa_max_khz")
    )

    fig, ax = plt.subplots(figsize=(10.5, 8), dpi=300)
    ax.scatter(agg["erro_d0"], agg["sep_d2_d0"], s=90, edgecolor="black", linewidth=0.8)

    for _, r in agg.iterrows():
        ax.text(r["erro_d0"], r["sep_d2_d0"], str(r["faixa_label"]).replace(" kHz", ""), fontsize=11, ha="left", va="bottom")

    ax.set_xlabel(f"Erro no sem dano — {metrica} D0", fontsize=23, labelpad=10)
    ax.set_ylabel(f"Separação de dano — {metrica} D2 − D0", fontsize=23)
    ax.set_title(
        f"{NOME_METODO.get(metodo, metodo)} — Pareto: erro baixo × separação alta em {metrica}",
        fontsize=24,
        pad=16,
    )
    estilo_eixos(ax)
    fig.tight_layout()
    salvar_fig(fig, PASTA_PARETO, f"pareto_erro_D0_vs_sep_D2_D0_{metodo}_{metrica}")
    plt.show()


# ============================================================
# 13) PARK — CÁLCULO OPCIONAL NAS MESMAS FAIXAS
# ============================================================

def shift_interp(x, f, tau):
    f_shift = f + tau
    return np.interp(f, f_shift, x, left=x[0], right=x[-1])


def park_single_extra(x, ref, fHz):
    df_band = fHz[-1] - fHz[0]
    tau_max = PARK_MAX_SHIFT_FRAC * df_band

    best_err = np.inf
    best_tau = 0.0
    best_dS = 0.0

    for tau in np.linspace(-tau_max, tau_max, PARK_NSTEPS):
        x_shift = shift_interp(x, fHz, tau)
        dS = np.mean(ref - x_shift)
        err = np.sum((ref - (x_shift + dS)) ** 2)
        if err < best_err:
            best_err = err
            best_tau = tau
            best_dS = dS

    y = shift_interp(x, fHz, best_tau) + best_dS
    y = moving_average(y, PARK_SMOOTH_WIN)
    return y


def compensar_park_extra(df, fcols, fHz):
    y_ref = curva_referencia_saudavel(df, fcols)
    X = df[fcols].to_numpy(float)
    Y = np.zeros_like(X)
    for i in range(len(X)):
        Y[i] = park_single_extra(X[i], y_ref, fHz)
    df2 = df.copy()
    df2[fcols] = Y
    return df2, y_ref


def calcular_metricas_compensadas(df_comp, fcols, y_ref):
    X = df_comp[fcols].to_numpy(float)
    out = df_comp[["temperatura_c", "falha"]].copy()
    out["RMSD"] = [rmsd(x, y_ref) for x in X]
    out["CCDM"] = [ccdm(x, y_ref) for x in X]
    return out


def rodar_park_larguras_ou_carregar(df_base):
    if os.path.exists(ARQ_METRICAS_PARK):
        print(f"✅ Carregando métricas Park existentes:\n{ARQ_METRICAS_PARK}")
        return preparar_metricas(pd.read_csv(ARQ_METRICAS_PARK))

    print("\n🔹 Calculando Park nas mesmas faixas acumuladas...")
    print("   Isso pode demorar, mas só roda uma vez; depois carrega o CSV salvo.")

    todas = []
    erros = []

    for fmin, fmax in FAIXAS_ACUMULADAS:
        try:
            fcols, fHz = get_freq_columns(df_base, fmin, fmax)
            if len(fcols) < 5:
                raise ValueError(f"Poucas colunas na faixa {fmin}-{fmax} kHz.")

            df_use = df_base[["temperatura_c", "falha"] + fcols].copy()
            print(f"\n🔸 Park | faixa {fmin:.0f}-{fmax:.0f} kHz | {len(fcols)} pontos")
            t0 = time.time()

            df_park_comp, y_ref = compensar_park_extra(df_use, fcols, fHz)
            df_met = calcular_metricas_compensadas(df_park_comp, fcols, y_ref)

            df_met["faixa_min_khz"] = float(fmin)
            df_met["faixa_max_khz"] = float(fmax)
            df_met["largura_khz"] = float(fmax - fmin)
            df_met["faixa_label"] = f"{int(fmin)}–{int(fmax)} kHz"
            df_met["n_freq_points"] = int(len(fcols))
            df_met["metodo"] = "Park"

            todas.append(df_met)
            print(f"✅ Park concluído em {time.time() - t0:.1f} s")

        except Exception as e:
            print(f"⚠️ Erro Park na faixa {fmin}-{fmax} kHz: {e}")
            erros.append({"faixa_min_khz": fmin, "faixa_max_khz": fmax, "erro": str(e)})

    if len(todas) == 0:
        raise RuntimeError("Nenhuma faixa Park foi processada.")

    df_park = preparar_metricas(pd.concat(todas, ignore_index=True))
    df_park.to_csv(ARQ_METRICAS_PARK, index=False)
    print(f"✅ Métricas Park salvas em:\n{ARQ_METRICAS_PARK}")

    if len(erros):
        pd.DataFrame(erros).to_csv(os.path.join(PASTA_PARK, "erros_park.csv"), index=False)

    return df_park


# ============================================================
# 14) COMPARAÇÃO RF × PARK
# ============================================================

def montar_comparacao_rf_park(df_sum):
    idx = [
        "faixa_min_khz", "faixa_max_khz", "largura_khz", "faixa_label",
        "temp_alvo_c", "temp_usada_c", "temp_label", "falha"
    ]
    rows = []

    for keys, g in df_sum.groupby(idx):
        fmin, fmax, largura, faixa_label, T_alvo, T_usada, T_label, falha = keys
        gm = g.set_index("metodo")
        if not all(m in gm.index for m in ["RF_ponto_a_ponto", "Park"]):
            continue

        rf = gm.loc["RF_ponto_a_ponto"]
        pk = gm.loc["Park"]

        rows.append({
            "faixa_min_khz": fmin,
            "faixa_max_khz": fmax,
            "largura_khz": largura,
            "faixa_label": faixa_label,
            "temp_alvo_c": T_alvo,
            "temp_usada_c": T_usada,
            "temp_label": T_label,
            "falha": int(falha),
            "RMSD_RF": float(rf["RMSD_medio"]),
            "RMSD_Park": float(pk["RMSD_medio"]),
            "CCDM_RF": float(rf["CCDM_medio"]),
            "CCDM_Park": float(pk["CCDM_medio"]),
            "delta_RMSD_RF_menos_Park": float(rf["RMSD_medio"] - pk["RMSD_medio"]),
            "delta_CCDM_RF_menos_Park": float(rf["CCDM_medio"] - pk["CCDM_medio"]),
            "RF_vence_RMSD": int(float(rf["RMSD_medio"]) < float(pk["RMSD_medio"])),
            "RF_vence_CCDM": int(float(rf["CCDM_medio"]) < float(pk["CCDM_medio"])),
        })

    return pd.DataFrame(rows)


def plot_delta_rf_park_limpo(df_comp, metrica="RMSD"):
    delta_col = f"delta_{metrica}_RF_menos_Park"
    if len(df_comp) == 0 or delta_col not in df_comp.columns:
        return

    for dano in DANOS_PLOTAR:
        sub = df_comp[df_comp["falha"] == dano].copy()
        faixas = ordem_faixas(sub)
        labels_x = faixas["faixa_label"].tolist()
        temps = sorted(sub["temp_alvo_c"].dropna().unique())
        labels_y = []
        for T in temps:
            ss = sub[np.isclose(sub["temp_alvo_c"], T)]
            labels_y.append(str(ss["temp_label"].iloc[0]) if len(ss) else format_temp_label(T))

        pivot = sub.pivot_table(index="temp_alvo_c", columns="faixa_label", values=delta_col, aggfunc="mean")
        pivot = pivot.reindex(index=temps, columns=labels_x)
        finite = pivot.values[np.isfinite(pivot.values)]
        vmax_abs = np.nanmax(np.abs(finite)) if len(finite) else 1.0

        fig, ax = plt.subplots(figsize=(max(11, 1.15 * len(labels_x) + 5), max(7, 0.52 * len(labels_y) + 3.5)), dpi=300)
        im = desenhar_heatmap(
            ax, pivot, labels_x, labels_y,
            titulo=f"RF − Park em {metrica} — dano {dano}\nnegativo = RF melhor | positivo = Park melhor",
            cmap="coolwarm",
            vmin=-vmax_abs,
            vmax=vmax_abs,
            texto=True,
        )
        ax.set_ylabel("Temperatura", fontsize=23, labelpad=10)
        cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
        cbar.set_label(f"RF − Park em {metrica}", fontsize=21)
        cbar.ax.tick_params(labelsize=16)
        fig.tight_layout()
        salvar_fig(fig, PASTA_PARK, f"delta_RF_menos_Park_{metrica}_dano{dano}")
        plt.show()


def plot_vitorias_rf_vs_park(df_comp):
    if len(df_comp) == 0:
        return

    tmp = df_comp.copy()
    tmp["RF_vence_total"] = tmp["RF_vence_RMSD"] + tmp["RF_vence_CCDM"]

    win = (
        tmp.groupby(["faixa_min_khz", "faixa_max_khz", "largura_khz", "faixa_label", "temp_alvo_c", "temp_label"], as_index=False)
        .agg(RF_vence_total=("RF_vence_total", "sum"))
    )

    faixas = ordem_faixas(win)
    labels_x = faixas["faixa_label"].tolist()
    temps = sorted(win["temp_alvo_c"].dropna().unique())
    labels_y = []
    for T in temps:
        ss = win[np.isclose(win["temp_alvo_c"], T)]
        labels_y.append(str(ss["temp_label"].iloc[0]) if len(ss) else format_temp_label(T))

    pivot = win.pivot_table(index="temp_alvo_c", columns="faixa_label", values="RF_vence_total", aggfunc="mean")
    pivot = pivot.reindex(index=temps, columns=labels_x)

    fig, ax = plt.subplots(figsize=(max(11, 1.15 * len(labels_x) + 5), max(7, 0.52 * len(labels_y) + 3.5)), dpi=300)
    im = desenhar_heatmap(
        ax, pivot, labels_x, labels_y,
        titulo="Vitórias do RF sobre Park\n0 = Park vence tudo | 6 = RF vence RMSD e CCDM nos três danos",
        cmap="viridis",
        vmin=0,
        vmax=6,
        texto=True,
    )
    ax.set_ylabel("Temperatura", fontsize=23, labelpad=10)
    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.025)
    cbar.set_label("Vitórias do RF", fontsize=21)
    cbar.ax.tick_params(labelsize=16)
    fig.tight_layout()
    salvar_fig(fig, PASTA_PARK, "vitorias_RF_vs_Park")
    plt.show()


def plot_barras_rf_vs_park_media(df_comp, metrica="RMSD"):
    if len(df_comp) == 0:
        return

    col_rf = f"{metrica}_RF"
    col_pk = f"{metrica}_Park"
    agg = (
        df_comp.groupby(["faixa_min_khz", "faixa_max_khz", "faixa_label"], as_index=False)
        .agg(RF=(col_rf, "mean"), Park=(col_pk, "mean"))
        .sort_values(["faixa_min_khz", "faixa_max_khz"])
    )

    x = np.arange(len(agg))
    bar_w = 0.36
    fig, ax = plt.subplots(figsize=(15, 8), dpi=300)
    ax.bar(x - bar_w / 2, agg["RF"], width=bar_w, edgecolor="black", linewidth=0.9, label="RF ponto a ponto")
    ax.bar(x + bar_w / 2, agg["Park"], width=bar_w, edgecolor="black", linewidth=0.9, label="Park")
    ax.set_xticks(x)
    ax.set_xticklabels(agg["faixa_label"], rotation=35, ha="right", fontsize=17)
    ax.set_ylabel(f"{metrica} médio", fontsize=23)
    ax.set_xlabel("Faixa analisada", fontsize=23, labelpad=10)
    ax.set_title(f"RF ponto a ponto × Park — {metrica} médio por faixa", fontsize=25, pad=16)
    ax.legend(frameon=True, fontsize=15)
    estilo_eixos(ax)
    fig.tight_layout()
    salvar_fig(fig, PASTA_PARK, f"barras_RF_vs_Park_{metrica}_medio")
    plt.show()


# ============================================================
# 15) CURVAS EXEMPLO
# ============================================================

def escolher_temperatura_disponivel(df_base, T_alvo):
    temps = sorted(df_base["temperatura_c"].dropna().unique())
    candidatos = []
    for T in temps:
        ok = True
        for d in DANOS_PLOTAR:
            if not np.any(np.isclose(df_base["temperatura_c"], T) & (df_base["falha"] == d)):
                ok = False
                break
        if ok:
            candidatos.append(float(T))

    if len(candidatos) == 0:
        raise RuntimeError("Nenhuma temperatura tem todos os danos para curvas exemplo.")

    candidatos = np.asarray(candidatos, dtype=float)
    return float(candidatos[np.argmin(np.abs(candidatos - float(T_alvo)))])


def escolher_faixa_curvas(df_score):
    if FAIXA_CURVAS_EXEMPLO is not None:
        return float(FAIXA_CURVAS_EXEMPLO[0]), float(FAIXA_CURVAS_EXEMPLO[1])

    rf = df_score[df_score["metodo"] == "RF_ponto_a_ponto"].copy()
    if len(rf) == 0:
        return FAIXAS_ACUMULADAS[len(FAIXAS_ACUMULADAS) // 2]

    best = (
        rf.groupby(["faixa_min_khz", "faixa_max_khz"], as_index=False)["score_pesquisa"]
        .mean()
        .sort_values("score_pesquisa")
        .iloc[0]
    )
    return float(best["faixa_min_khz"]), float(best["faixa_max_khz"])


def plot_curvas_exemplo_rf_park(df_base, df_score):
    if not GERAR_CURVAS_EXEMPLO:
        return

    try:
        fmin, fmax = escolher_faixa_curvas(df_score)
        fcols, fHz = get_freq_columns(df_base, fmin, fmax)
        if len(fcols) < 5:
            print("⚠️ Poucas colunas para curvas exemplo.")
            return

        df_use = df_base[["temperatura_c", "falha"] + fcols].copy()
        print(f"\n🔹 Recalculando RF/Park para curvas exemplo na faixa {fmin:.0f}–{fmax:.0f} kHz...")
        df_rf_comp, y_ref_rf = compensar_rf_direto(df_use, fcols)
        df_park_comp, y_ref_pk = compensar_park_extra(df_use, fcols, fHz)

        fkhz = np.asarray(fHz, dtype=float) / 1e3
        temps_usadas = []
        for T_alvo in TEMPS_CURVAS_EXEMPLO:
            T = escolher_temperatura_disponivel(df_base, T_alvo)
            if not any(np.isclose(T, t) for t in temps_usadas):
                temps_usadas.append(T)

        for T in temps_usadas:
            fig, axes = plt.subplots(1, len(DANOS_PLOTAR), figsize=(7.5 * len(DANOS_PLOTAR), 7), dpi=300, sharey=True)
            if len(DANOS_PLOTAR) == 1:
                axes = [axes]

            for ax, dano in zip(axes, DANOS_PLOTAR):
                mask = np.isclose(df_use["temperatura_c"], T) & (df_use["falha"] == dano)
                idxs = np.where(mask.to_numpy())[0]
                if len(idxs) == 0:
                    ax.set_title(f"Dano {dano}\nsem amostra", fontsize=22)
                    continue

                idx = int(idxs[0])
                y_orig = df_use.iloc[idx][fcols].to_numpy(float)
                y_rf = df_rf_comp.iloc[idx][fcols].to_numpy(float)
                y_pk = df_park_comp.iloc[idx][fcols].to_numpy(float)

                ax.plot(fkhz, y_ref_rf, "--", color="black", linewidth=1.8, label=f"Referência {REF_TEMP}°C")
                ax.plot(fkhz, y_orig, color="tab:red", alpha=0.55, linewidth=1.5, label=f"Original {format_temp_num(T)}°C")
                ax.plot(fkhz, y_rf, color="tab:blue", linewidth=2.4, label="RF ponto a ponto")
                ax.plot(fkhz, y_pk, color="tab:green", linewidth=2.2, label="Park")
                ax.set_title(f"Dano {dano}", fontsize=24, pad=14)
                ax.set_xlabel("Frequência (kHz)", fontsize=22, labelpad=10)
                estilo_eixos(ax)

            axes[0].set_ylabel("Impedância", fontsize=23)
            handles, labels = axes[0].get_legend_handles_labels()
            fig.legend(handles, labels, loc="lower center", ncol=4, frameon=True, fontsize=14, bbox_to_anchor=(0.5, -0.08))
            fig.suptitle(
                f"Curvas exemplo — RF × Park — {fmin:.0f}–{fmax:.0f} kHz — T = {format_temp_num(T)}°C",
                fontsize=27,
                y=1.02,
            )
            fig.tight_layout(rect=[0, 0.08, 1, 0.95])
            nome = f"curvas_exemplo_RF_Park_{fmin:.0f}_{fmax:.0f}kHz_T{format_temp_num(T).replace('-', 'm')}C"
            salvar_fig(fig, PASTA_CURVAS, nome)
            plt.show()

    except Exception as e:
        print(f"⚠️ Não consegui gerar curvas exemplo: {e}")


# ============================================================
# 16) EXECUÇÃO GERAL
# ============================================================

def rodar_graficos_pesquisa_extras_v2():
    print("\n" + "=" * 100)
    print("GRÁFICOS EXTRAS V2 — RF / FAIXAS / TEMPERATURA / PARK")
    print("=" * 100)

    # -----------------------------
    # RF
    # -----------------------------
    df_rf = carregar_metricas_rf()
    if len(df_rf) == 0:
        raise RuntimeError("Nenhuma métrica RF válida encontrada para as faixas acumuladas.")

    # Coloca temperaturas alvo (-10 a 80) usando temperatura real mais próxima.
    df_rf_plot, mapa_rf = aplicar_temperaturas_alvo(df_rf)
    df_sum_rf = resumo_metricas(df_rf_plot)

    path_sum_rf = os.path.join(PASTA_EXTRA, "resumo_metricas_RF_temperaturas_alvo.csv")
    df_sum_rf.to_csv(path_sum_rf, index=False)
    print(f"✅ Resumo RF salvo em: {path_sum_rf}")

    # Heatmaps principais.
    print("\n📊 Heatmaps limpos RF...")
    for metrica in METRICAS_PLOTAR:
        plot_heatmap_metrica_triptico(df_sum_rf, metrica=metrica, metodo="RF_ponto_a_ponto")
        plot_heatmap_metrica_individual(df_sum_rf, metrica=metrica, metodo="RF_ponto_a_ponto")

    # Separação, contraste e inversões.
    print("\n📊 Separação e contraste RF...")
    df_sep_rf = calcular_tabela_separacao(df_sum_rf)
    path_sep_rf = os.path.join(PASTA_SEPARACAO, "separacoes_RF.csv")
    df_sep_rf.to_csv(path_sep_rf, index=False)
    print(f"✅ Separações RF salvas em: {path_sep_rf}")

    for metrica in METRICAS_PLOTAR:
        plot_heatmap_separacao_individual(df_sep_rf, metrica=metrica, metodo="RF_ponto_a_ponto")
        plot_heatmap_contraste_relativo(df_sep_rf, metrica=metrica, metodo="RF_ponto_a_ponto")

    plot_mapa_inversoes(df_sep_rf, metodo="RF_ponto_a_ponto")

    # Score e ranking.
    print("\n📊 Score e ranking RF...")
    df_score_rf = montar_score_pesquisa(df_sep_rf)
    path_score_rf = os.path.join(PASTA_RANKING, "score_pesquisa_RF.csv")
    df_score_rf.to_csv(path_score_rf, index=False)
    print(f"✅ Score RF salvo em: {path_score_rf}")

    plot_score_heatmap(df_score_rf, metodo="RF_ponto_a_ponto")
    rank_rf = plot_ranking_medio_faixas(df_score_rf, metodo="RF_ponto_a_ponto")
    plot_melhor_faixa_por_temperatura(df_score_rf)

    # Robustez térmica.
    print("\n📊 Robustez térmica RF...")
    df_sens_rf = calcular_sensibilidade_termica(df_sum_rf)
    path_sens_rf = os.path.join(PASTA_ROBUSTEZ, "sensibilidade_termica_RF.csv")
    df_sens_rf.to_csv(path_sens_rf, index=False)
    print(f"✅ Sensibilidade térmica RF salva em: {path_sens_rf}")

    for metrica in METRICAS_PLOTAR:
        plot_linhas_metricas_por_faixa(df_sum_rf, metrica=metrica, metodo="RF_ponto_a_ponto")
        plot_sensibilidade_termica(df_sens_rf, metrica=metrica, metodo="RF_ponto_a_ponto")
        plot_coef_variacao_por_faixa(df_sum_rf, metrica=metrica, metodo="RF_ponto_a_ponto")
        plot_pareto_erro_vs_separacao(df_score_rf, metrica=metrica, metodo="RF_ponto_a_ponto")

    df_metricas_todos_plot = df_rf_plot.copy()
    df_sum_todos = df_sum_rf.copy()
    df_sep_todos = df_sep_rf.copy()
    df_score_todos = df_score_rf.copy()

    # -----------------------------
    # Park opcional
    # -----------------------------
    if GERAR_COMPARACAO_PARK:
        print("\n📊 Comparação RF × Park...")
        try:
            df_base = carregar_base_extra()
            df_park = rodar_park_larguras_ou_carregar(df_base)
            df_park_plot, mapa_park = aplicar_temperaturas_alvo(df_park)

            df_metricas_todos_plot = preparar_metricas(pd.concat([df_rf, df_park], ignore_index=True))
            df_metricas_todos_plot, mapa_todos = aplicar_temperaturas_alvo(df_metricas_todos_plot)
            df_sum_todos = resumo_metricas(df_metricas_todos_plot)

            path_sum_todos = os.path.join(PASTA_EXTRA, "resumo_metricas_RF_e_Park_temperaturas_alvo.csv")
            df_sum_todos.to_csv(path_sum_todos, index=False)
            print(f"✅ Resumo RF+Park salvo em: {path_sum_todos}")

            # Gráficos Park também, no mesmo padrão.
            for metrica in METRICAS_PLOTAR:
                plot_heatmap_metrica_triptico(df_sum_todos, metrica=metrica, metodo="Park")
                plot_heatmap_metrica_individual(df_sum_todos, metrica=metrica, metodo="Park")

            df_sep_todos = calcular_tabela_separacao(df_sum_todos)
            df_score_todos = montar_score_pesquisa(df_sep_todos)
            path_score_todos = os.path.join(PASTA_RANKING, "score_pesquisa_RF_e_Park.csv")
            df_score_todos.to_csv(path_score_todos, index=False)
            print(f"✅ Score RF+Park salvo em: {path_score_todos}")

            for metrica in METRICAS_PLOTAR:
                plot_heatmap_separacao_individual(df_sep_todos, metrica=metrica, metodo="Park")
                plot_heatmap_contraste_relativo(df_sep_todos, metrica=metrica, metodo="Park")
                plot_linhas_metricas_por_faixa(df_sum_todos, metrica=metrica, metodo="Park")
                df_sens_todos = calcular_sensibilidade_termica(df_sum_todos)
                plot_sensibilidade_termica(df_sens_todos, metrica=metrica, metodo="Park")
                plot_coef_variacao_por_faixa(df_sum_todos, metrica=metrica, metodo="Park")
                plot_pareto_erro_vs_separacao(df_score_todos, metrica=metrica, metodo="Park")

            plot_mapa_inversoes(df_sep_todos, metodo="Park")
            plot_score_heatmap(df_score_todos, metodo="Park")
            plot_ranking_medio_faixas(df_score_todos, metodo="Park")
            plot_melhor_faixa_por_temperatura(df_score_todos)

            # Delta RF - Park.
            df_comp = montar_comparacao_rf_park(df_sum_todos)
            path_comp = os.path.join(PASTA_PARK, "comparacao_RF_vs_Park_temperaturas_alvo.csv")
            df_comp.to_csv(path_comp, index=False)
            print(f"✅ Comparação RF vs Park salva em: {path_comp}")

            for metrica in METRICAS_PLOTAR:
                plot_delta_rf_park_limpo(df_comp, metrica=metrica)
                plot_barras_rf_vs_park_media(df_comp, metrica=metrica)

            plot_vitorias_rf_vs_park(df_comp)

            if GERAR_CURVAS_EXEMPLO:
                plot_curvas_exemplo_rf_park(df_base, df_score_todos)

        except Exception as e:
            print(f"⚠️ Não consegui gerar comparação Park: {e}")
            print("   Os gráficos RF foram gerados normalmente.")

    else:
        print("\n⚠️ GERAR_COMPARACAO_PARK = False. Comparação RF × Park desativada.")

    print("\n" + "=" * 100)
    print("✅ TODOS OS GRÁFICOS EXTRAS V2 FORAM GERADOS")
    print(f"📁 Pasta principal: {PASTA_EXTRA}")
    print("=" * 100)

    return df_metricas_todos_plot, df_sum_todos, df_sep_todos, df_score_todos


# ============================================================
# 17) EXECUTAR
# ============================================================

if __name__ == "__main__":
    df_metricas_pesquisa_v2, df_sum_pesquisa_v2, df_sep_pesquisa_v2, df_score_pesquisa_v2 = rodar_graficos_pesquisa_extras_v2()
